<a href="https://colab.research.google.com/github/rilianx/BilinearReduction/blob/main/ReduceBilinears_paper.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!wget -O pybex1.9.2.tar.gz 'https://drive.google.com/uc?id=1A_XPZnDn6jZjLMA2zMpxL2yo44Ky-X1s'  &> /dev/null
!pip install pybex pybex1.9.2.tar.gz
!gdown https://drive.google.com/uc?id=1lS5q0100EgUZYMjetYPy7jf4TG6o_JgR
!chmod +x ibexopt

!git clone https://github.com/Aluzinus727/affine-arithmetic.git
%cd affine-arithmetic/
from solvers.af2 import AF2
%cd /content/

Processing ./pybex1.9.2.tar.gz
  Preparing metadata (setup.py) ... done
  Created wheel for pyibex: filename=pyibex-1.9.0-cp312-cp312-linux_x86_64.whl size=4439766 sha256=d4c39344cd48fc3c8cf5217e27535a7fe6d5ad7b07b460e0c0dea0bd85b58935
  Stored in directory: /tmp/pip-ephem-wheel-cache-xbc4tz85/wheels/44/94/1f/b08e664955277d45148a19f8a5c709e465a524833436ad116a
Successfully built pyibex
  Attempting uninstall: pyibex
    Found existing installation: pyibex 1.9.0
    Uninstalling pyibex-1.9.0:
      Successfully uninstalled pyibex-1.9.0
Downloading...
From: https://drive.google.com/uc?id=1lS5q0100EgUZYMjetYPy7jf4TG6o_JgR
To: /content/ibexopt
100% 4.46M/4.46M [00:00<00:00, 173MB/s]
fatal: destination path 'affine-arithmetic' already exists and is not an empty directory.
/content/affine-arithmetic
/content


In [2]:
import subprocess, re
import os

#args = '-r 1e-7 -f \"10*x[0] + 2*x[1] - x[2] + 2*x[3] - 2*x[4] + x[0]*x[5] + 2*x[1]*x[5] + x[2]*x[5] + 2*x[3]*x[5] + x[4]*x[5] - 10*x[0]*x[5]^2 + 2*x[1]*x[5]^2 - x[2]*x[5]^2 + x[3]*x[5]^2 + 3*x[4]*x[5]^2\" -x \"x[6] in [0,1];\"'

def bounds(vars, xl, x, ctrs=None, prec=1e-4):

  def write_vars(f):
    f.write('variables\n')
    f.write(vars+' in (')
    for i in range(len(x)):
      if i == len(x)-1:
        f.write(str(x[i]))
      else:
        f.write(str(x[i])+'; ')
    f.write(');\n\n')

  #minimization
  # Abre el archivo en modo escritura
  with open('tmp.txt', 'w') as f:
    write_vars(f)

    f.write('minimize '+xl+';')
    if ctrs is not None:
      f.write('\nconstraints\n')
      for ctr in ctrs: f.write(ctr+'\n')
      f.write("end\n")


  p = subprocess.Popen(["./ibexopt", '-r', str(prec), '-a', str(prec), '-t', '5', 'tmp.txt'], stdout=subprocess.PIPE)

  for line in p.communicate()[0].decode().splitlines():
    #print(line)
    if line.find("f* in")!=-1:
      lb = float(re.search(r'\[(.*),(.*)\]', line).groups(0)[1])
    if line.find("number of cells:")!=-1:
      n_cells = int(re.search('\t\t([0-9]*)', line).groups(0)[0])

  #maximization
  with open('tmp.txt', 'w') as f:
    write_vars(f)

    f.write('minimize -('+xl+');\n')
    if ctrs is not None:
      f.write('\nconstraints\n')
      for ctr in ctrs: f.write(ctr+'\n')
      f.write("end\n")


  p = subprocess.Popen(["./ibexopt", '-r', str(prec), '-a', str(prec), '-t', '5', 'tmp.txt'], stdout=subprocess.PIPE)


  for line in p.communicate()[0].decode().splitlines():
    #print(line)
    if line.find("f* in")!=-1:
      ub = -float(re.search(r'\[(.*),(.*)\]', line).groups(0)[0])
    if line.find("number of cells:")!=-1:
      n_cells += int(re.search('\t\t([0-9]*)', line).groups(0)[0])


  return (lb,ub), n_cells




#!./ibexopt -r 1e-4 tmp.txt


In [3]:
import random
from pyibex import Interval, IntervalVector

def new_af2_term(interval, id):
    a, b = interval
    base = (a + b) / 2
    coeff = (b - a) / 2
    new_epsilons = {id: coeff}

    return AF2(
        base=base,
        epsilons=new_epsilons
    )

def get_error_af2(af2_expr):
    total_error = af2_expr.non_affine*Interval(-1,1)+af2_expr.non_affine_p*Interval(0,1)+af2_expr.non_affine_n*Interval(-1,0)
    return total_error.diam()



def random_interval(min_val, max_val, min_width, max_width):
    width = random.uniform(min_width, max_width)
    start = random.uniform(min_val, max_val - width)
    end = start + width
    return Interval(start, end)

def random_domains(
    n_vars=3,
    min_val=0,
    max_val=10,
    min_width=0.5,
    max_width=2.0
):
    """
    Genera un conjunto de dominios aleatorios (IntervalVector) y sus términos AF2 correspondientes.

    Parámetros:
        n_vars (int): número de variables a generar.
        min_val (float): límite inferior posible de los intervalos.
        max_val (float): límite superior posible de los intervalos.
        min_width (float): ancho mínimo de los intervalos.
        max_width (float): ancho máximo de los intervalos.

    Retorna:
        global_x (IntervalVector): vector de intervalos aleatorios.
        afx_terms (list): lista de términos AF2 generados.
    """
    # Crear los intervalos aleatorios
    intervals = [
        random_interval(min_val, max_val, min_width, max_width)
        for _ in range(n_vars)
    ]

    # Crear el IntervalVector
    global_x = IntervalVector(intervals)

    # Crear los términos AF2
    afx_terms = [
        new_af2_term(global_x[i], i + 1)
        for i in range(n_vars)
    ]

    return (global_x, *afx_terms)

In [4]:
import numpy as np
from dataclasses import dataclass
from typing import List, Tuple # Added for type hinting in ReductionResult

# Moved the definition of ReductionResult here so it's available for to_expression
@dataclass
class ReductionResult:
    Q_res: np.ndarray
    blocks: int
    initial_offdiag_norm: float
    final_offdiag_norm: float
    initial_offdiag_max: float
    final_offdiag_max: float
    cap_final: np.ndarray
    pos_indices: List[int]
    neg_indices: List[int]
    full_cancels: int
    cap_limited: int
    residual_terms: List[Tuple[int, int, int, float, float]]
    total_delta_e: float

def to_expression(res: ReductionResult, var_prefix="x"):
   expr = quadratic_form_str(res.Q_res, var_prefix)
   for term in res.residual_terms:
       sign, i, j, a, b, sign2 = term
       #assuming a=b, otherwise (a*x_i + b*x_j)^2
       if a != b:
           expr += f" + {sign}({a}*{var_prefix}_{i+1} + {"-" if not sign2 else ""}{b}*{var_prefix}_{j+1})**2"
       else:
           expr += f" + {sign}{a*a}*({var_prefix}_{i+1} + {"-" if not sign2 else ""}{var_prefix}_{j+1})**2"

   # Limpieza estética: eliminar '+ -' si aparecen
   expr = expr.replace("+ -", "- ")
   return expr


def quadratic_form_str(Q, var_prefix="x"):
    """
    Devuelve la expresión simbólica del cuadrático x^T Q x
    en formato tipo:
        x_1**2 + 1.6*x_1*x_2 + ...

    Parámetros:
        Q (np.ndarray): matriz cuadrada (simétrica preferiblemente)
        var_prefix (str): prefijo de las variables (por defecto 'x')

    Retorna:
        str: expresión simbólica expandida
    """
    Q = np.array(Q, dtype=float)
    n = Q.shape[0]

    terms = []

    # Términos cuadráticos (diagonales)
    for i in range(n):
        coef = Q[i, i]
        if coef != 0:
            terms.append(f"{coef}*{var_prefix}_{i+1}**2" if coef != 1 else f"{var_prefix}_{i+1}**2")

    # Términos cruzados (solo parte superior de la matriz)
    for i in range(n):
        for j in range(i+1, n):
            coef = Q[i, j] + Q[j, i]  # sumar simétricos
            if coef != 0:
                terms.append(f"{coef}*{var_prefix}_{i+1}*{var_prefix}_{j+1}")

    # Unir todo con +
    expr = " + ".join(terms)
    # Limpieza estética: eliminar '+ -' si aparecen
    expr = expr.replace("+ -", "- ")

    return expr


### Reduce Bilinears (algorithm)

In [5]:
import numpy as np
import math
import heapq
from typing import Optional

def reduce_bilinears_double_knapsack(Q_in: np.ndarray,
                                     tol: Optional[float] = 1e-12,
                                     verbose: bool = False,
                                     early_stop: bool = True,
                                     w: Optional[np.ndarray] = None) -> ReductionResult:
    """
    Reduce términos bilineales de Q usando "Doble Mochila" Simétrica con Lazy Max-Heap.
    Alineado con el algoritmo "Greedy Diagonal Budgeting":
      - Extracción simétrica: alpha = min(|Q_ij|, |Q_ii|, |Q_jj|).
      - Ordenamiento guiado por reducción de error: Delta E = alpha * w_i * w_j.
      - Optimización con Max-Heap y lazy updates para complejidad O(m log m).
    """

    # --- Validaciones y preparación ---
    Q_in = np.asarray(Q_in, dtype=float)
    if Q_in.ndim != 2 or Q_in.shape[0] != Q_in.shape[1]:
        raise ValueError("Q debe ser una matriz cuadrada real.")
    Q = symmetrize(Q_in)
    n = Q.shape[0]
    total_delta_e = 0

    if tol is None:
        tol = 1e-12 * max(1.0, np.linalg.norm(Q, ord='fro'))
    tol = max(float(tol), 0.0)

    # Si no se proveen anchos de intervalo, asumimos 1.0 (ordenamiento por magnitud pura)
    if w is None:
        w = np.ones(n, dtype=float)

    # Índices y capacidades iniciales
    pos_idx = [i for i in range(n) if Q[i, i] > tol]
    neg_idx = [i for i in range(n) if Q[i, i] < -tol]

    cap = np.zeros(n, dtype=float)
    for i in pos_idx: cap[i] = Q[i, i]
    for j in neg_idx: cap[j] = -Q[j, j]

    if verbose:
        print_subheader("Mochilas iniciales (Simétricas)")
        print(f"Positivas (idx) : {pos_idx}")
        print(f"Negativas (idx) : {neg_idx}")

    initial_off = offdiag_fro_norm(Q)
    initial_max = offdiag_max_abs(Q)

    # --- Construcción inicial del Max-Heap ---
    # En Python heapq es min-heap, así que guardamos -delta_E
    heap = []

    # Función local para evaluar y pushear pares válidos
    def push_to_heap(i: int, j: int, group_name: str, diag_sign: float):
        mag = abs(Q[i, j])
        if mag > tol:
            alpha = min(mag, cap[i], cap[j])
            delta_e = 0.5*alpha * w[i] * w[j]
            if delta_e > tol:
                heapq.heappush(heap, (-delta_e, i, j, group_name, diag_sign))

    # Llenar heap con pares pos-pos
    for a in range(len(pos_idx)):
        for b in range(a + 1, len(pos_idx)):
            push_to_heap(pos_idx[a], pos_idx[b], "+", 1.0)

    # Llenar heap con pares neg-neg
    for a in range(len(neg_idx)):
        for b in range(a + 1, len(neg_idx)):
            push_to_heap(neg_idx[a], neg_idx[b], "-", -1.0)

    blocks = 0
    full_cancels = 0
    cap_limited = 0
    residual_terms = []

    # --- Bucle Principal (Lazy Evaluation) ---
    while heap:
        neg_delta_e, i, j, group, diag_sign = heapq.heappop(heap)
        stored_delta_e = -neg_delta_e

        # Recalcular alpha y delta_E con las capacidades actuales
        mag = abs(Q[i, j])
        alpha = min(mag, cap[i], cap[j])
        curr_delta_e = 0.5*alpha * w[i] * w[j]

        # Si ya no hay presupuesto, descartar
        if curr_delta_e <= tol:
            continue

        # Tolerancia para la comparación flotante de "igualdad"
        if abs(curr_delta_e - stored_delta_e) <= 1e-15:
            # ¡Máximo global confirmado! Aplicar extracción.
            qij_sign = math.copysign(1.0, Q[i, j])

            # Actualizar matriz y capacidades
            Q[i, j] -= qij_sign * alpha
            Q[j, i] = Q[i, j]
            Q[i, i] -= diag_sign * alpha
            Q[j, j] -= diag_sign * alpha
            cap[i] -= alpha
            cap[j] -= alpha

            total_delta_e += curr_delta_e

            blocks += 1
            if abs(Q[i, j]) <= tol:
                full_cancels += 1
            else:
                cap_limited += 1

            # Guardar el término residual (a_val = b_val = sqrt(alpha) por simetría)
            val = math.sqrt(alpha)
            sign2 = (qij_sign == diag_sign)
            residual_terms.append((group, i, j, val, val, sign2))

            if verbose:
                print(f"[{group}] ({i},{j}) ext: {alpha:.6f} | cap_i: {cap[i]:.4f}, cap_j: {cap[j]:.4f} | Qij -> {Q[i,j]:+.6f}")

            if early_stop and offdiag_max_abs(Q) <= tol:
                break
        else:
            # Lazy update: el valor bajó, pero no es cero, lo devolvemos al heap
            heapq.heappush(heap, (-curr_delta_e, i, j, group, diag_sign))

    # --- Finalización ---
    Q = symmetrize(Q)
    final_off = offdiag_fro_norm(Q)
    final_max = offdiag_max_abs(Q)

    if verbose:
        print_subheader("Resumen reducción (Simétrica + Heap)")
        print(f"Bloques aplicados     : {blocks}")
        print(f"Full cancels          : {full_cancels}")
        print(f"Cap-limited           : {cap_limited}")
        print(f"||Q||_off (antes->des): {initial_off:.6f} -> {final_off:.6f}")
        print(Q)
        print(residual_terms)

    return ReductionResult(
        Q_res=Q,
        blocks=blocks,
        initial_offdiag_norm=initial_off,
        final_offdiag_norm=final_off,
        initial_offdiag_max=initial_max,
        final_offdiag_max=final_max,
        cap_final=cap.copy(),
        pos_indices=pos_idx,
        neg_indices=neg_idx,
        full_cancels=full_cancels,
        cap_limited=cap_limited,
        residual_terms=residual_terms,
        total_delta_e=total_delta_e
    )

## Experimento

In [6]:
import re

def expr_to_minibex(expr: str) -> str:
    # 1️⃣ Reemplazar ** → ^
    expr = expr.replace("**", "^")

    # 2️⃣ Reemplazar x_i → x[i-1]
    def repl(match):
        idx = int(match.group(1)) - 1
        return f"x[{idx}]"

    expr = re.sub(r"\bx_(\d+)\b", repl, expr)

    return expr

In [7]:
def make_random_Q(n: int,
                  density: float = 0.5,
                  diag_pos_ratio: float = 0.5,
                  rng: Optional[np.random.Generator] = None,
                  allow_near_zero_diag: bool = False,
                  near_zero_frac: float = 0.0,
                  near_zero_eps: float = 1e-8,
                  max_diag: float = 1.0) -> np.ndarray:
    """
    Genera una matriz simétrica aleatoria n x n.
    - density: probabilidad de término off-diagonal no nulo.
    - diag_pos_ratio: fracción de diagonales positivas.
    - allow_near_zero_diag: si True, inyecta algunas diagonales en [-eps, eps].
    """
    if rng is None:
        rng = np.random.default_rng(12345)

    Q = np.zeros((n, n), dtype=float)

    # Definir qué diagonales serán positivas y negativas
    if diag_pos_ratio <= 0:
        pos_idx = []
        neg_idx = list(range(n))
    elif diag_pos_ratio >= 1:
        pos_idx = list(range(n))
        neg_idx = []
    else:
        pos_count = int(round(diag_pos_ratio * n))
        pos_idx = list(rng.choice(np.arange(n), size=pos_count, replace=False))
        neg_idx = [i for i in range(n) if i not in pos_idx]

    # Asignar valores de diagonal (evitar ~0 por defecto)
    for i in pos_idx:
        Q[i, i] = rng.uniform(0., max_diag)
    for j in neg_idx:
        Q[j, j] = -rng.uniform(0., max_diag)

    # Inyectar diagonales casi cero si se desea testear bordes
    if allow_near_zero_diag and near_zero_frac > 0.0:
        k = max(0, min(n, int(round(near_zero_frac * n))))
        idxs = rng.choice(np.arange(n), size=k, replace=False)
        for t in idxs:
            Q[t, t] = rng.uniform(-near_zero_eps, near_zero_eps)

    # Asignar off-diagonales con la densidad especificada
    for i in range(n):
        for j in range(i + 1, n):
            if rng.random() < density:
                val = rng.uniform(-1.0, 1.0)
            else:
                val = 0.0
            Q[i, j] = val
            Q[j, i] = val

    return Q

def symmetrize(Q: np.ndarray) -> np.ndarray:
    """Asegura simetría exacta numéricamente: Q = (Q + Q^T)/2."""
    return 0.5 * (Q + Q.T)

def offdiag_fro_norm(Q: np.ndarray) -> float:
    r"""
    Norma de Frobenius de los elementos fuera de la diagonal:
    ||Q||_off = sqrt( sum_{i!= j} Q_ij^2 ).
    """
    n = Q.shape[0]
    if n == 0:
        return 0.0
    mask = ~np.eye(n, dtype=bool)
    return float(np.linalg.norm(Q[mask]))

def offdiag_max_abs(Q: np.ndarray) -> float:
    """Máximo |Q_ij| con i!= j (para ver el peor término bilineal residual)."""
    n = Q.shape[0]
    if n <= 1:
        return 0.0
    mask = ~np.eye(n, dtype=bool)
    vals = np.abs(Q[mask])
    return float(vals.max()) if vals.size else 0.0

In [8]:
import numpy as np
import sympy as sp
import contextlib
import io
falso_stdout = io.StringIO()


def experimento_af(
    tamanios=[3, 5, 8, 10],
    n_experimentos=10,
    min_val=0, max_val=10,
    min_width=0.5, max_width=2.0,
    density=0.7, diag_pos_ratio=1.0, max_diag=1.0,
    tol=None, verbose=False
):
    """
    Compara el error afín de la expresión original vs la modificada
    para distintos tamaños de matrices cuadráticas.

    Parámetros:
    -----------
    tamanios : list[int]
        Lista con los tamaños de las matrices Q a probar.
    n_experimentos : int
        Número de instancias aleatorias por tamaño.
    min_val, max_val, min_width, max_width : float
        Parámetros para la generación de dominios.
    density, diag_pos_ratio : float
        Parámetros de make_random_Q.
    tol : float | None
        Tolerancia para reduce_bilinears_double_knapsack.
    verbose : bool
        Si True, muestra los resultados parciales.

    Retorna:
    --------
    resultados : dict
        Diccionario con los errores promedio y desviaciones por tamaño.
    """

    resultados = {}

    for n in tamanios:
        errores_orig = []
        errores_mod = []

        for i in range(n_experimentos):
            # === Generar dominios y matriz ===
            global_x, *xs = random_domains(
                n_vars=n, min_val=min_val, max_val=max_val,
                min_width=min_width, max_width=max_width
            )
            for j, xj in enumerate(xs, 1):
              globals()[f"x_{j}"] = xj
            Q = make_random_Q(n, density=density, diag_pos_ratio=diag_pos_ratio, max_diag=max_diag)

            # === Construir expresiones ===
            expr = quadratic_form_str(Q)

            expr_modificada = to_expression(
                reduce_bilinears_double_knapsack(Q, tol=tol, verbose=False, w=np.array(global_x.diam()))
            )

            expr_minibex = expr_to_minibex(expr)


            # === Evaluar linearizaciones ===
            with contextlib.redirect_stdout(falso_stdout):
              af_expr = eval(expr)
              af_expr_mod = eval(expr_modificada)


            expr_lineal=""
            for i in range(n):
              expr_lineal += f"{af_expr.epsilons[i+1]/global_x[i].rad()} * x[{i}]+"
            #remove last character
            expr_lineal = expr_lineal[:-1]

            intercepts,_ = bounds(f"x[{n}]", f"{expr_minibex}-({expr_lineal})", global_x, prec=1e-8)

            real_range= intercepts[1]-intercepts[0]


            err_orig = get_error_af2(af_expr)/real_range
            err_mod = get_error_af2(af_expr_mod)/real_range

            errores_orig.append(err_orig)
            errores_mod.append(err_mod)


            if verbose:
                print(f"\n[n={n}] experimento {i+1}: error_orig={err_orig:.4e}, error_mod={err_mod:.4e}")

        resultados[n] = {
            "mean_original": np.mean(errores_orig),
            "std_original": np.std(errores_orig),
            "mean_ours": np.mean(errores_mod),
            "std_ours": np.std(errores_mod),
        }

    return resultados

In [ ]:
experimento_af(
    tamanios=[3, 5, 10],
    n_experimentos=30,
    min_val=0, max_val=10,
    min_width=0.1, max_width=0.2,
    density=0.5, diag_pos_ratio=1.0, max_diag=1.0,
    tol=None, verbose=False
)

In [ ]:
experimento_af(
    tamanios=[3, 5, 10],
    n_experimentos=30,
    min_val=0, max_val=10,
    min_width=0.1, max_width=0.2,
    density=0.2, diag_pos_ratio=1.0,
    tol=None, verbose=False
)

In [ ]:
experimento_af(
    tamanios=[3, 5, 10],
    n_experimentos=30,
    min_val=0, max_val=10,
    min_width=0.1, max_width=0.2,
    density=0.9, diag_pos_ratio=1.0,
    tol=None, verbose=False
)

In [ ]:
experimento_af(
    tamanios=[3, 5, 10],
    n_experimentos=30,
    min_val=-10, max_val=10,
    min_width=0.1, max_width=0.2,
    density=0.5, diag_pos_ratio=0.5,
    tol=None, verbose=False
)

In [ ]:
experimento_af(
    tamanios=[3, 5, 10],
    n_experimentos=10,
    min_val=-10, max_val=10,
    min_width=0.1, max_width=0.2,
    density=0.5, diag_pos_ratio=1.0,
    tol=None, verbose=False, max_diag=4.0
)

In [ ]:
experimento_af(
    tamanios=[3, 5, 10],
    n_experimentos=10,
    min_val=-10, max_val=10,
    min_width=0.1, max_width=0.2,
    density=0.5, diag_pos_ratio=1.0,
    tol=None, verbose=False, max_diag=0.5
)

In [9]:
!pip install pipreqs


INFO: Not scanning for jupyter notebooks.


In [10]:
!pipreqs /content

INFO: Not scanning for jupyter notebooks.
<unknown>:24: SyntaxWarning: invalid escape sequence '\['
<unknown>:43: SyntaxWarning: invalid escape sequence '\['
Please, verify manually the final list of requirements.txt to avoid possible dependency confusions.
Please, verify manually the final list of requirements.txt to avoid possible dependency confusions.
INFO: Successfully saved requirements file in /content/requirements.txt
